In [ ]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import re
import nltk
from nltk.corpus import  stopwords
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [48]:
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('stopwords')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\manis\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\manis\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\manis\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [49]:
df = pd.read_csv("D:/mannu/ML/resume screener/dataset/Resume/Resume.csv")

In [50]:
df.head()

,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR


In [51]:
df.drop(columns=['ID', 'Resume_html'], axis=1, inplace=True)

In [52]:
df.head(5)

,Resume_str,Category
0,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,HR
1,"HR SPECIALIST, US HR OPERATIONS ...",HR
2,HR DIRECTOR Summary Over 2...,HR
3,HR SPECIALIST Summary Dedica...,HR
4,HR MANAGER Skill Highlights ...,HR


In [53]:
df["Resume_str"][0]

"         HR ADMINISTRATOR/MARKETING ASSOCIATE\n\nHR ADMINISTRATOR       Summary     Dedicated Customer Service Manager with 15+ years of experience in Hospitality and Customer Service Management.   Respected builder and leader of customer-focused teams; strives to instill a shared, enthusiastic commitment to customer service.         Highlights         Focused on customer satisfaction  Team management  Marketing savvy  Conflict resolution techniques     Training and development  Skilled multi-tasker  Client relations specialist           Accomplishments      Missouri DOT Supervisor Training Certification  Certified by IHG in Customer Loyalty and Marketing by Segment   Hilton Worldwide General Manager Training Certification  Accomplished Trainer for cross server hospitality systems such as    Hilton OnQ  ,   Micros    Opera PMS   , Fidelio    OPERA    Reservation System (ORS) ,   Holidex    Completed courses and seminars in customer service, sales strategies, inventory control, loss pr

In [54]:
df["Category"].value_counts()

Category
INFORMATION-TECHNOLOGY    120
BUSINESS-DEVELOPMENT      120
ADVOCATE                  118
CHEF                      118
ENGINEERING               118
ACCOUNTANT                118
FINANCE                   118
FITNESS                   117
AVIATION                  117
SALES                     116
BANKING                   115
HEALTHCARE                115
CONSULTANT                115
CONSTRUCTION              112
PUBLIC-RELATIONS          111
HR                        110
DESIGNER                  107
ARTS                      103
TEACHER                   102
APPAREL                    97
DIGITAL-MEDIA              96
AGRICULTURE                63
AUTOMOBILE                 36
BPO                        22
Name: count, dtype: int64

In [55]:
df.shape

(2484, 2)

In [56]:
new_df = df[df["Category"]=='INFORMATION-TECHNOLOGY']

In [57]:
new_df.shape

(120, 2)

In [58]:
new_df.reset_index(drop=True, inplace=True)

In [59]:
new_df.sample(5)

,Resume_str,Category
50,DIRECTOR OF INFORMATION TECHNOLOGY ...,INFORMATION-TECHNOLOGY
41,INFORMATION TECHNOLOGY SPECIALIST ...,INFORMATION-TECHNOLOGY
26,DIRECTOR OF INFORMATION TECHNOLOGY ...,INFORMATION-TECHNOLOGY
15,INFORMATION TECHNOLOGY SPECIALIST ...,INFORMATION-TECHNOLOGY
63,INFORMATION TECHNOLOGY COORDINATOR ...,INFORMATION-TECHNOLOGY


In [60]:
stop_words = set(stopwords.words('english'))

PROTECTED_WORDS = {
    'it', 'ms', 'sql', 'aws', 'ai', 'ml', 'ui', 'ux',
    'c', 'c++', 'c#'
}

def transform_text(text):
    text = re.sub('[^a-zA-Z0-9]+#', ' ',text)

    text = text.lower()
    
    words = text.split()

    cleaned_words = []
    for word in words:
        if word in PROTECTED_WORDS:
            cleaned_words.append(word)
            continue
        
        if word in stop_words:
            continue
        if len(word) <= 2 and word not in ['ai', 'ml', 'ms']:
            continue
        cleaned_words.append(lemmatizer.lemmatize(word))
    
    return ' '.join(cleaned_words)

    

In [61]:
new_df["clean_resume"]= new_df["Resume_str"].apply(transform_text)

In [62]:
new_df["clean_resume"]

0      information technology summary dedicated infor...
1      information technology specialist gs11 experie...
2      information technology supervisor summary seek...
3      information technology instructor summary seve...
4      information technology manager/analyst profess...
                             ...                        
115    corporate project manager career overview seas...
116    it technology specialist professional summary ...
117    it manager highlight customer client relation ...
118    subject matter expert (information technology ...
119    training manager executive summary qualified t...
Name: clean_resume, Length: 120, dtype: object

In [63]:
original_text = new_df["Resume_str"].iloc[0]

In [64]:
transform__text = new_df["clean_resume"].iloc[0]

In [65]:
print(original_text[:900])
print(transform__text[:900])


         INFORMATION TECHNOLOGY         Summary     Dedicated  Information Assurance Professional  well-versed in analyzing and mitigating risk and finding cost-effective solutions. Excels at boosting performance and productivity by establishing realistic goals and enforcing deadlines.  Versatile IT professional with 37 years of Enterprise design and engineering methodology.       Skills          Enterprise platforms  Knowledge of Product Lifecycle Management (PLM)  Project tracking  Hardware and software upgrade planning  Product requirements documentation  Self-directed  MS Visio  Decisive  Collaborative  Domain Active Directory Layout  Data storage engineering      Information Assurance  Risk Management Framework (RMF)  Active Directory design and deployment  Workstation build and deployment  Systems Accreditation Packages  Red Hat Enterprise Linux installation and hardening  Network 
information technology summary dedicated information assurance professional well-versed analyzing m

# Tf -IDF encoding

In [66]:
corpus = new_df["clean_resume"].tolist()

job_description = """
We are hiring an Information Technology Engineer responsible for designing,
developing, and maintaining software systems. The candidate should have strong
experience in python programming, sql databases, data structures and algorithms,
and rest api development. Hands-on experience with linux, git version control,
and cloud platforms such as aws is required. Familiarity with system design,
debugging, and basic networking concepts is a plus.
"""


In [67]:
corpus

["information technology summary dedicated information assurance professional well-versed analyzing mitigating risk finding cost-effective solutions. excels boosting performance productivity establishing realistic goal enforcing deadlines. versatile it professional year enterprise design engineering methodology. skill enterprise platform knowledge product lifecycle management (plm) project tracking hardware software upgrade planning product requirement documentation self-directed ms visio decisive collaborative domain active directory layout data storage engineering information assurance risk management framework (rmf) active directory design deployment workstation build deployment system accreditation package red hat enterprise linux installation hardening network design troubleshooting high performance computing experience company name city state information technology 02/2011 current hired manage accreditation effort major department modernization project involving accreditation pac

In [80]:
def rank_resumes(clean_resumes, clean_jd, top_n=5):
    Tfidf = TfidfVectorizer(
        max_features=3000,
        ngram_range = (1,2)
    )

    resume_vectors = Tfidf.fit_transform(corpus)

    jd_vector = Tfidf.transform([job_description])

    similarity_score = cosine_similarity(jd_vector, resume_vectors).flatten()
    top_indices = np.argsort(scores)[::-1][:top_n]

    return top_indices, similarity_score


## Rank resumes

In [ ]:
# scores = similarity_score.flatten() # convert 2D into 1D 
# top_indices = np.argsort(scores)[::-1][:5] # 1. asencending order and return descending and take top 5 indices

# for i in top_indices:
#     print(f"Score: {scores[i]:.3f}")
#     print(df.iloc[i]['Resume_str'][:200])
#     print("------")

Score: 0.119
         FIELD HR ASSOCIATE           Summary    Reliable HR Field Associate with a Master's of science in Human Resource management emphasis as a Generalist. Passionate and motivated with a drive for
------
Score: 0.102
         GRAPHIC DESIGNER           Summary    Enthusiastic student majoring in Chemistry; great at performing many task in a timely matter and as efficient as possible. Strong background in computer 
------
Score: 0.101
         E-LEARNING DESIGNER           Career Overview    Highly skilled and experienced educator with a strong
background in information technology. Adept at addressing the needs of a variety of lear
------
Score: 0.097
         HR SERVICES REPRESENTATIVE       Summary     A multi-skilled professional with good all-round HR imformatory skills. Very capable with an ability deal with all the recruitment/processing need
------
Score: 0.095
         DESIGNER / TECHNICAL DESIGNER           Summary     Creative fashion designer with background 

“I computed cosine similarity between the job description vector and all resume vectors, then sorted the scores to retrieve the top matching candidates.”

## -> Skill gap analysis

In [73]:
SKILLS = {
    "python", "java", "sql", "aws", "linux", "git",
    "rest api", "api", "docker", "kubernetes",
    "data structures", "algorithms",
    "system design", "networking"
}

clean_jd = transform_text(job_description)

In [74]:
def extract_skills(text, skills):
    found_skills = set()
    for skill in SKILLS:
        if skill in text:
            found_skills.add(skill)
    return found_skills

In [75]:
jd_skills = extract_skills(clean_jd, SKILLS)
print("JD skills:" , jd_skills)

JD skills: {'api', 'networking', 'sql', 'linux', 'aws', 'algorithms', 'system design', 'git', 'python', 'rest api'}


In [76]:
resume_text = new_df.iloc[top_indices[0]]["clean_resume"]
resume_skills = extract_skills(resume_text, SKILLS)

missing_skills = jd_skills - resume_skills

print("Resume skills :" , resume_skills)
print("missing skills :" , missing_skills)


Resume skills : {'aws', 'networking', 'python', 'linux'}
missing skills : {'api', 'sql', 'algorithms', 'system design', 'git', 'rest api'}


combining ranking + skill gap

In [78]:
display_socre = scores * 100

In [79]:
for i in top_indices[:3]:
    resume_text = new_df.iloc[i]['clean_resume']
    resume_skills = extract_skills(resume_text, SKILLS)
    missing = jd_skills - resume_skills

    print(f"Score: {display_socre[i]:.3f}")
    print("missing skills: ", missing)
    print("---------------------------")

Score: 11.950
missing skills:  {'api', 'sql', 'algorithms', 'system design', 'git', 'rest api'}
---------------------------
Score: 10.199
missing skills:  {'api', 'sql', 'aws', 'algorithms', 'system design', 'git', 'python', 'rest api'}
---------------------------
Score: 10.054
missing skills:  {'api', 'sql', 'linux', 'aws', 'algorithms', 'system design', 'git', 'rest api'}
---------------------------


Typical ranges in real text systems:

0.00 – 5 → almost no match

5 – 12 → weak but relevant match

12 – 25 → decent match

< 30 → very strong (rare in resumes)

format & normalize output

In [81]:
def get_skill_gap(jd_skills, resume_skills):
    """
    return skills missing in resume compare to jd
    """
    return jd_skills - rresume_skillse